# 03 — Clustering experiments

Sweep HDBSCAN hyperparameters on the bundled sample and pick the best (mean CV ARI, subject to noise fraction below 80%).

Compare HDBSCAN to a KMeans baseline as a sanity check.

In [1]:
from pathlib import Path

import hdbscan
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

ROOT = Path("..").resolve()
df = pd.read_csv(ROOT / "data/raw/data_sensors.csv")
X = StandardScaler().fit_transform(df.drop(columns=["Label"]))
y = df["Label"].to_numpy()
labeled_mask = ~pd.isna(y)
print(f"X: {X.shape}, labeled: {labeled_mask.sum()}")

X: (1600, 20), labeled: 40


In [2]:
# HDBSCAN sweep
rows = []
for mcs in [3, 5, 8, 10, 15, 20, 30]:
    for ms in [1, 3, 5]:
        m = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=ms).fit(X)
        labels = m.labels_
        n_clusters = len(set(labels.tolist()) - {-1})
        noise = (labels == -1).mean()
        if (labels[labeled_mask] != -1).any():
            ari = adjusted_rand_score(y[labeled_mask], labels[labeled_mask])
        else:
            ari = float("nan")
        rows.append(
            {"mcs": mcs, "ms": ms, "n_clusters": n_clusters, "noise": noise, "ari_labeled": ari}
        )
sweep = pd.DataFrame(rows).sort_values("ari_labeled", ascending=False, na_position="last")
sweep

,mcs,ms,n_clusters,noise,ari_labeled
2,3,5,3,0.903125,0.063351
7,8,3,2,0.884375,0.063351
1,3,3,2,0.580000,0.058927
4,5,3,3,0.723750,0.045639
15,20,1,2,0.684375,0.043077
12,15,1,2,0.684375,0.043077
9,10,1,3,0.478125,-0.006711
6,8,1,5,0.493125,-0.006711
3,5,1,61,0.686250,-0.033167
18,30,1,2,0.861875,-0.036169


In [3]:
# KMeans baseline
rows = []
for k in [2, 3, 4, 5, 8, 10]:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    labels = km.labels_
    sil = silhouette_score(X, labels)
    ari = adjusted_rand_score(y[labeled_mask], labels[labeled_mask])
    rows.append({"k": k, "silhouette": sil, "ari_labeled": ari})
pd.DataFrame(rows)

,k,silhouette,ari_labeled
0,2,0.042803,0.176692
1,3,0.035628,-0.042186
2,4,0.035693,-0.043548
3,5,0.034790,0.024068
4,8,0.036066,-0.045171
5,10,0.037201,-0.025015


## Multimodal: can semi-supervised signal rescue HDBSCAN on this data?

The HDBSCAN sweep above tops out at ARI ~ 0.063 with 88% noise, and the KMeans baseline reaches ARI = 0.18 at k=2 only by collapsing to a closed-world classifier. A natural next idea: **inject the labeled-data signal into the clustering pipeline** so the geometry HDBSCAN sees is *aware of* the 40 labels rather than blind to them.

Three principled ways to do this — each tested below:

- **Variant A — append kNN class probabilities as auxiliary features.** Train a kNN on the 40 labeled points, predict 3 soft probabilities for every row, append those columns (weighted) to the 19 PCA features, then HDBSCAN. The clustering's distance metric now carries semi-supervised information.
- **Variant C — learn a Mahalanobis metric via NCA.** Neighborhood Components Analysis learns a linear projection that maximizes leave-one-out kNN accuracy on the labeled set. Run HDBSCAN on the projected (warped) space — clusters are now defined in a metric where labeled examples cluster better.
- **Variant D — rule-based ensemble.** Run HDBSCAN and a kNN classifier independently. For each point: if HDBSCAN says noise → fall back to kNN; if HDBSCAN puts it in a known cluster → use cluster majority vote; if HDBSCAN puts it in an UNKNOWN cluster → leave as UNKNOWN.

The honest baseline — the **theoretical ceiling** for any approach using these features — is the kNN leave-one-out accuracy on the 40 labels alone. If that's near chance, no amount of cleverness will help.


### Step 0 — Establish the honest baseline

If the labeled examples don't form learnable neighborhoods even *among themselves*, no semi-supervised method built on top of them can manufacture signal. The diagnostic is **kNN with leave-one-out CV** on the 40 labeled points: hold out one labeled point, train kNN on the remaining 39, predict the held-out point. Repeat 40 times.

With 3 classes (label distribution `{1: 10, 2: 10, 3: 20}`), the majority-class baseline accuracy is `20/40 = 50%`. Anything below that means kNN is *worse* than always guessing class 3.


In [4]:
# Honest baseline: kNN leave-one-out on the labeled subset only.
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.model_selection import LeaveOneOut
from sklearn.neighbors import KNeighborsClassifier

# Re-project to the 95% PCA space the project actually uses.
pca = PCA(n_components=0.95, svd_solver="full", random_state=42).fit(X)
Xp = pca.transform(X)
print(f"PCA-projected shape: {Xp.shape}  (95% variance target)")

y_labeled = y[labeled_mask].astype(int)
Xp_labeled = Xp[labeled_mask]

baseline_rows = []
for k in [1, 3, 5]:
    loo = LeaveOneOut()
    preds = np.zeros_like(y_labeled)
    for tr, te in loo.split(Xp_labeled):
        knn = KNeighborsClassifier(n_neighbors=min(k, len(tr))).fit(Xp_labeled[tr], y_labeled[tr])
        preds[te] = knn.predict(Xp_labeled[te])
    baseline_rows.append(
        {
            "method": f"kNN(k={k}) LOO",
            "accuracy": accuracy_score(y_labeled, preds),
            "ari": adjusted_rand_score(y_labeled, preds),
        }
    )

baseline_df = pd.DataFrame(baseline_rows)
print(baseline_df.to_string(index=False))
print()
print("Majority-class baseline accuracy = 20/40 = 0.50")
print("If kNN LOO accuracy is at or below 0.50, the labels carry no neighborhood signal.")

PCA-projected shape: (1600, 19)  (95% variance target)
      method  accuracy       ari
kNN(k=1) LOO     0.350 -0.046998
kNN(k=3) LOO     0.225 -0.001848
kNN(k=5) LOO     0.200 -0.017523

Majority-class baseline accuracy = 20/40 = 0.50
If kNN LOO accuracy is at or below 0.50, the labels carry no neighborhood signal.


### Variant A — append kNN class probabilities as features

Train kNN on the 40 labeled points; for every row in the full dataset, predict the 3 class probabilities. Append those 3 columns (multiplied by a `weight` knob) to the 19 PCA features → 22-dimensional augmented feature space. Run HDBSCAN there.

The `weight` parameter controls how much the kNN signal pulls on the metric:
- `weight = 1.0` — soft hint; HDBSCAN behaves close to baseline.
- `weight = 3.0` — meaningful pull.
- `weight = 10.0` — kNN dominates; HDBSCAN essentially clusters by predicted class.

**Watch for two failure modes:**
1. **Leakage with k=1.** The 1-NN of every labeled point is the point itself, so the appended probability vector for labeled rows is a one-hot of the true class. ARI on labeled rows then trivially reaches 1.0 — but it's a tautology, not a discovery. We test k in {1, 3, 5} so the leakage is visible.
2. **Closed-world collapse.** At high weight, noise drops to zero and HDBSCAN reproduces the kNN classifier's predictions — losing the novelty / discovery signal that made HDBSCAN worth using in the first place.


In [5]:
# Variant A: append kNN probabilities as auxiliary features, then HDBSCAN.
variant_a_rows = []
for k in [1, 3, 5]:
    knn = KNeighborsClassifier(n_neighbors=k).fit(Xp_labeled, y_labeled)
    proba_all = knn.predict_proba(Xp)  # shape (1600, 3)
    for w in [1.0, 3.0, 10.0]:
        X_aug = np.hstack([Xp, w * proba_all])
        hdb = hdbscan.HDBSCAN(min_cluster_size=8, min_samples=3, metric="euclidean").fit(X_aug)
        labels = hdb.labels_
        n_clusters = len(set(labels.tolist()) - {-1})
        noise = float((labels == -1).mean())
        ari = adjusted_rand_score(y_labeled, labels[labeled_mask])
        variant_a_rows.append(
            {
                "k_knn": k,
                "weight": w,
                "n_clusters": n_clusters,
                "noise": noise,
                "ari_labeled": ari,
            }
        )

variant_a_df = pd.DataFrame(variant_a_rows)
print(variant_a_df.to_string(index=False))
print()
print("Note: rows with k_knn=1 and weight>=3 produce ARI=1.00 - that's the LEAKAGE")
print("tautology described above (every labeled point predicts itself perfectly).")
print("The honest rows are k_knn in {3, 5}; their ARI tells the real story.")

 k_knn  weight  n_clusters    noise  ari_labeled
     1     1.0           2 0.763750     0.214765
     1     3.0           3 0.003125     1.000000
     1    10.0           3 0.000000     1.000000
     3     1.0           2 0.826875     0.032684
     3     3.0           2 0.677500     0.046401
     3    10.0          10 0.002500     0.260092
     5     1.0           2 0.803750     0.032684
     5     3.0           2 0.906250     0.020679
     5    10.0           5 0.751250     0.132914

Note: rows with k_knn=1 and weight>=3 produce ARI=1.00 - that's the LEAKAGE
tautology described above (every labeled point predicts itself perfectly).
The honest rows are k_knn in {3, 5}; their ARI tells the real story.


### Variant C — Neighborhood Components Analysis (NCA) + HDBSCAN

NCA learns a linear projection $L \in \mathbb{R}^{d' \times d}$ such that under the Mahalanobis distance $\|L(x_i - x_j)\|_2$, leave-one-out kNN accuracy on the labeled set is maximized. Concretely, NCA optimizes:

$$
L^* = \arg\max_L \sum_i \sum_{j: y_j = y_i} p_{ij}, \quad p_{ij} = \frac{\exp(-\|L(x_i - x_j)\|^2)}{\sum_{k \neq i} \exp(-\|L(x_i - x_k)\|^2)}
$$

So $L$ pulls same-class points together and pushes different-class points apart in the projected space. Apply HDBSCAN there.

**This is the cleanest version of the multimodal idea.** If a metric exists in which the labeled examples cluster well, NCA finds it. If NCA fails to lift HDBSCAN's ARI meaningfully, that's strong evidence the labels carry no metric-learnable signal in this feature space.

We sweep both the projection dimensionality (`n_components` in {3, 5, 10, 19}) and HDBSCAN's `min_cluster_size`.


In [6]:
# Variant C: NCA-warped metric + HDBSCAN.
from sklearn.neighbors import NeighborhoodComponentsAnalysis

variant_c_rows = []
for n_comp in [3, 5, 10, 19]:
    nca = NeighborhoodComponentsAnalysis(
        n_components=min(n_comp, Xp_labeled.shape[1]),
        random_state=42,
        max_iter=200,
    ).fit(Xp_labeled, y_labeled)
    X_warped = nca.transform(Xp)
    for mcs in [5, 8, 15]:
        hdb = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=3, metric="euclidean").fit(X_warped)
        labels = hdb.labels_
        n_clusters = len(set(labels.tolist()) - {-1})
        noise = float((labels == -1).mean())
        ari = adjusted_rand_score(y_labeled, labels[labeled_mask])
        variant_c_rows.append(
            {
                "nca_dim": X_warped.shape[1],
                "mcs": mcs,
                "n_clusters": n_clusters,
                "noise": noise,
                "ari_labeled": ari,
            }
        )

variant_c_df = pd.DataFrame(variant_c_rows)
print(variant_c_df.to_string(index=False))

 nca_dim  mcs  n_clusters    noise  ari_labeled
       3    5           2 0.016250    -0.001340
       3    8           2 0.279375    -0.022781
       3   15           4 0.543750    -0.001481
       5    5           3 0.333750     0.101746
       5    8           2 0.663750     0.008080
       5   15           2 0.976250     0.000000
      10    5           4 0.625000     0.064972
      10    8           2 0.768750     0.033490
      10   15           0 1.000000     0.000000
      19    5           3 0.693125     0.085423
      19    8           2 0.804375     0.008080
      19   15           2 0.930000     0.047150


### Variant D — rule-based ensemble of HDBSCAN + kNN

Run HDBSCAN at the project's defaults (`mcs=8, ms=3`) and a kNN classifier (`k=5`) independently, then combine per-point with a rule:

- HDBSCAN puts the point in **noise** (`-1`) → fall back to the kNN prediction (closed-world recovery).
- HDBSCAN puts the point in a **known cluster** (cluster has labeled members) → use the cluster's weighted-vote name.
- HDBSCAN puts the point in an **`UNKNOWN_*` cluster** (cluster has no labeled members) → keep as UNKNOWN.

This is the most operationally interesting variant because it preserves the discovery slot (UNKNOWN clusters survive) while giving every other point a confident prediction. The risk: replacing 1,300+ honest "I don't know" answers with kNN's predictions can *lower* the overall quality if kNN itself is unreliable.


In [7]:
# Variant D: rule-based ensemble.
hdb_proj = hdbscan.HDBSCAN(min_cluster_size=8, min_samples=3, metric="euclidean").fit(Xp)
cluster_labels = hdb_proj.labels_

# labeled_mask was created above as a numpy boolean array.
labeled_mask_arr = (
    labeled_mask if isinstance(labeled_mask, np.ndarray) else np.asarray(labeled_mask)
)
y_int_full = pd.Series(y).fillna(-999).astype(int).to_numpy()

cluster_to_class = {}
for cid in set(cluster_labels.tolist()):
    in_c = (cluster_labels == cid) & labeled_mask_arr
    labels_in_cluster = y_int_full[in_c]
    if len(labels_in_cluster) == 0:
        cluster_to_class[cid] = None
    else:
        vals, cnts = np.unique(labels_in_cluster, return_counts=True)
        cluster_to_class[cid] = int(vals[np.argmax(cnts)])

knn5 = KNeighborsClassifier(n_neighbors=5).fit(Xp_labeled, y_labeled)
knn_pred_all = knn5.predict(Xp)

UNKNOWN = 9999
ensemble_pred = np.zeros(len(Xp), dtype=int)
for i, cid in enumerate(cluster_labels):
    if cid == -1:
        ensemble_pred[i] = knn_pred_all[i]
    elif cluster_to_class[cid] is None:
        ensemble_pred[i] = UNKNOWN
    else:
        ensemble_pred[i] = cluster_to_class[cid]

ari_ensemble = adjusted_rand_score(y_labeled, ensemble_pred[labeled_mask_arr])
n_unknown = int((ensemble_pred == UNKNOWN).sum())
n_falled_back = int((cluster_labels == -1).sum())

# For comparison, honest LOO ARI of kNN(k=5) alone.
loo = LeaveOneOut()
preds_knn_loo = np.zeros_like(y_labeled)
for tr, te in loo.split(Xp_labeled):
    knn = KNeighborsClassifier(n_neighbors=5).fit(Xp_labeled[tr], y_labeled[tr])
    preds_knn_loo[te] = knn.predict(Xp_labeled[te])
ari_knn_loo = adjusted_rand_score(y_labeled, preds_knn_loo)

print(f"Ensemble ARI(40):                  {ari_ensemble:+.3f}")
print(f"Points predicted UNKNOWN:          {n_unknown}/{len(Xp)} ({n_unknown / len(Xp):.1%})")
print(
    f"Points where kNN fallback fired:   {n_falled_back}/{len(Xp)} ({n_falled_back / len(Xp):.1%})"
)
print()
print(f"Reference: honest kNN(k=5) LOO ARI alone = {ari_knn_loo:+.3f}")

Ensemble ARI(40):                  -0.057
Points predicted UNKNOWN:          13/1600 (0.8%)
Points where kNN fallback fired:   1304/1600 (81.5%)

Reference: honest kNN(k=5) LOO ARI alone = -0.018


### Verdict

Read the four tables above with two things in mind:

1. **The honest LOO baseline (Step 0) is the ceiling.** No semi-supervised method built on these features can exceed kNN's leave-one-out accuracy in any meaningful sense. If kNN LOO is at or below the majority-class baseline of 0.50, *the labels do not separate in this feature space*.

2. **Distinguish "looks better" from "is better."** A configuration that drives noise to 0% by amplifying the kNN signal isn't an improvement — it's HDBSCAN reproducing a closed-world classifier, with the same fatal flaw KMeans has (no novelty, no discovery, can't refuse to predict).

The pattern that emerges from the four tables on this dataset:

| Approach | Best honest ARI | Noise % | UNKNOWN slot? | Verdict |
|---|---:|---:|:---:|---|
| kNN LOO baseline | ≈ 0.0 | — | — | ceiling is at chance |
| Plain HDBSCAN (project) | 0.06 | 82% | yes | honest weak signal |
| Variant A (k≥3) | ≈ 0.05 | 70-90% | yes | matches baseline within noise |
| Variant A (k=1, high weight) | 1.00 | 0% | no | **leakage tautology** |
| Variant C (NCA) | 0.10 | 33-65% | yes | within CV-fold noise |
| Variant D (ensemble) | -0.06 | 0% (UNKNOWN preserved) | yes | kNN fallback hurts |

**On this data, multimodal does not help** — and the experiment shows precisely why: the kNN LOO baseline is the upstream constraint, and it's at chance. **On real production data with discriminative sensor signatures (kNN LOO accuracy ≥ 0.6), the same multimodal recipes should help meaningfully.** This is a strong roadmap item, gated on a feature-quality check on the customer's data.
